In [1]:
import pandas as pd
import requests

In [2]:
with open("API_KEY.txt", "r") as f:
    FRED_API_KEY = f.read().strip()


In [3]:
def fetch_fred_series(series_id, api_key):
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    observations = response.json()["observations"]

    df = pd.DataFrame(observations)[["date", "value"]]
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")  # "." -> NaN
    return df

In [16]:
oil_data = fetch_fred_series("DCOILWTICO", FRED_API_KEY)
cpi_data = fetch_fred_series("CPIAUCSL", FRED_API_KEY)     

For the sake of consistent analysis, we will have the dates start Jan 1 1980 and end March 1. CPI is backward looking for the past month, and so will our oil volatility calculation. All we need to ensure is that null values are managed correctly and the data will line up.

In [20]:
from datetime import datetime

start = datetime(1980, 1, 1)
end = datetime(2023, 3, 1)

oil_data = oil_data[(oil_data["date"] >= start) & (oil_data["date"] <= end)]
cpi_data = cpi_data[(cpi_data["date"] >= start) & (cpi_data["date"] <= end)]

In [26]:
cpi_data.isnull().sum()

date     0
value    0
dtype: int64

In [27]:
oil_data.isnull().sum()

date       0
value    332
dtype: int64

In [28]:
oil_data = oil_data.dropna(subset=["value"])

In [29]:
oil_data.isnull().sum()

date     0
value    0
dtype: int64

The rest of the file from here is some exploratory data analysis.

In [13]:
oil_data.tail()

,date,value
10513,2026-04-21,93.64
10514,2026-04-22,94.76
10515,2026-04-23,99.27
10516,2026-04-24,98.42
10517,2026-04-27,99.89


In [14]:
cpi_data.tail()

,date,value
946,2025-11-01,325.063
947,2025-12-01,326.031
948,2026-01-01,326.588
949,2026-02-01,327.460
950,2026-03-01,330.293


In [7]:
avg_price = oil_data['DCOILWTICO'].mean()
oil_data_clean = oil_data.fillna({'DCOILWTICO': avg_price})

In [13]:
oil_data.columns = ['DATE', 'OIL_PRICE']
cpi_data.columns = ['DATE', 'CPI']


In [17]:
oil_mean = oil_data['OIL_PRICE'].mean()
oil_min = oil_data['OIL_PRICE'].min()
oil_max = oil_data['OIL_PRICE'].max()
oil_median = oil_data['OIL_PRICE'].median()
oil_std = oil_data['OIL_PRICE'].std()
print(oil_mean)
print(oil_min)
print(oil_max)
print(oil_median)
print(oil_std)

77.21844675740593
55.44
123.64
75.03
12.820987663547518


In [18]:
cpi_mean = cpi_data['CPI'].mean()
cpi_min = cpi_data['CPI'].min()
cpi_max = cpi_data['CPI'].max()
cpi_median = cpi_data['CPI'].median()
cpi_std = cpi_data['CPI'].std()

print(cpi_mean)
print(cpi_min)
print(cpi_max)
print(cpi_median)
print(cpi_std)

0.2973834891894962
-1.9152895328595805
1.80586907449211
0.29055444645697703
0.3558606076134019
